# Tugas Hands on 3, Menganalisis data dari case employee attrition.

Nama : Anisa Ansuya Henna

Kelas : TIF RP 24 C CID

NIM   : 24552011020

In [119]:
!pip install pandas numpy matplotlib seaborn scikit-learn

DATASET

In [120]:
import pandas as pd
import numpy as np
import os

file_path = 'dataset_karyawan/case5_employee_attrition.csv'

if os.path.exists(file_path):
    df_mentah = pd.read_csv(file_path)
    print(f"Berhasil memuat dataset dari: {file_path}")
else:
    print(f"Error: File tidak ditemukan di '{file_path}'. Silakan unggah file ke folder 'dataset_karyawan'.")


if 'df_mentah' in locals() or 'df_mentah' in globals():
    print("Ukuran Data:", df_mentah.shape)
    df_mentah.info()

Berhasil memuat dataset dari: dataset_karyawan/case5_employee_attrition.csv
Ukuran Data: (1200, 17)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 17 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   employee_id                   1200 non-null   object 
 1   usia                          1170 non-null   float64
 2   jenis_kelamin                 1172 non-null   object 
 3   departemen                    1169 non-null   object 
 4   jabatan                       1182 non-null   object 
 5   bidang_pendidikan             1152 non-null   object 
 6   lama_di_perusahaan_tahun      1176 non-null   float64
 7   lama_di_jabatan_tahun         1153 non-null   float64
 8   gaji_bulanan_idr              1161 non-null   float64
 9   lembur                        1162 non-null   object 
 10  kepuasan_kerja                1165 non-null   float64
 11  keseimbangan_hidup_ke

1. Mengecek kualitas data
2. Membuang data kembar
3. Mengisi missing value
4. Menangani data aneh (Outlier) menggunakan teknik penghapusan
5. Kemudian membuat tabel perbandingan sebelum vs sesudah

In [121]:
def cek_kualitas(d, kolom_num, kolom_grup, kunci):
    g = d.groupby(kolom_grup, observed=True)[kolom_num]
    q1 = g.transform(lambda s: s.quantile(0.25))
    q3 = g.transform(lambda s: s.quantile(0.75))
    iqr = q3 - q1
    outlier = (d[kolom_num] < q1 - 1.5 * iqr) | (d[kolom_num] > q3 + 1.5 * iqr)
    return pd.Series({
        'n_baris': len(d),
        'n_duplikat': d.duplicated(subset=kunci).sum(),
        'n_missing': d.isnull().sum().sum(),
        'n_outlier': outlier.sum(),
    })

if 'df_mentah' not in locals() and 'df_mentah' not in globals():
    print("Error: Variabel 'df_mentah' belum terdefinisi.")
    print("Silakan pastikan file 'case5_employee_attrition.csv' sudah diunggah dan sel kedua dijalankan dengan sukses terlebih dahulu.")
else:
    df_bersih = df_mentah.drop_duplicates(subset=['employee_id']).copy()

    df_bersih.fillna(df_bersih.median(numeric_only=True), inplace=True)

    laporan = pd.DataFrame({
        'sebelum': cek_kualitas(df_mentah, 'gaji_bulanan_idr', 'jabatan', 'employee_id'),
        'sesudah': cek_kualitas(df_bersih, 'gaji_bulanan_idr', 'jabatan', 'employee_id'),
    })
    print(laporan)

            sebelum  sesudah
n_baris        1200     1200
n_duplikat        0        0
n_missing       574      163
n_outlier         0        0


Analisis Rata-rata kepuasan kerja

In [122]:
analisis_1 = df_bersih.groupby(['departemen', 'keluar_perusahaan'], observed=True)['kepuasan_kerja'].mean()
print("Analisis 1 (Rata-rata kepuasan kerja):\n", analisis_1)

analisis_2 = df_bersih.groupby(['lembur', 'keluar_perusahaan'], observed=True)['gaji_bulanan_idr'].mean()
print("\nAnalisis 2 (Rata-rata gaji bulanan):\n", analisis_2)

Analisis 1 (Rata-rata kepuasan kerja):
 departemen  keluar_perusahaan
Finance     Tidak                2.694118
            Ya                   1.904110
HR          Tidak                2.863309
            Ya                   2.438596
IT          Tidak                2.722892
            Ya                   2.400000
Operations  Tidak                2.687500
            Ya                   2.368421
Sales       Tidak                2.632979
            Ya                   2.089744
Name: kepuasan_kerja, dtype: float64

Analisis 2 (Rata-rata gaji bulanan):
 lembur  keluar_perusahaan
Tidak   Tidak                3.412162e+07
        Ya                   3.144388e+07
Ya      Tidak                3.482125e+07
        Ya                   3.558769e+07
Name: gaji_bulanan_idr, dtype: float64



Split data 70% training dan 30% test. Kemudian file output dibuat


In [123]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import os

X = df_bersih.drop(columns=['keluar_perusahaan', 'employee_id'])
y = df_bersih['keluar_perusahaan']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

kolom_angka = X.select_dtypes(include=['int64', 'float64']).columns
kolom_teks = X.select_dtypes(include=['object', 'category']).columns

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), kolom_angka),
        ('cat', OneHotEncoder(handle_unknown='ignore'), kolom_teks)
    ]
)

X_train_final = preprocessor.fit_transform(X_train)
X_test_final = preprocessor.transform(X_test)

os.makedirs('output', exist_ok=True)

df_bersih.to_csv('output/data_bersih.csv', index=False)

kamus_data = pd.DataFrame({
    'kolom': df_bersih.columns,
    'tipe_data': df_bersih.dtypes.values,
    'n_missing': df_bersih.isnull().sum().values,
    'n_unik': df_bersih.nunique().values
})
kamus_data.to_csv('output/kamus_data.csv', index=False)

if 'laporan' in locals() or 'laporan' in globals():
    laporan.to_csv('laporan_kualitas.csv')

print("Semua file berhasil disimpan di folder output!")

Semua file berhasil disimpan di folder output!
